# 安装 sentence_transformers
uv pip install --python "D:\Code\LLM\.venv\Scripts\python.exe" \
  --index-url https://pypi.tuna.tsinghua.edu.cn/simple \
  sentence-transformers

# 安装 ipykernel
uv pip install --python "D:\Code\LLM\.venv\Scripts\python.exe" --index-url https://pypi.tuna.tsinghua.edu.cn/simple ipykernel 2>&1 | tail -15

# 安装 pymilvus
uv pip install --python "D:\Code\LLM\.venv\Scripts\python.exe" --index-url https://pypi.tuna.tsinghua.edu.cn/simple <包名>

# 拉取模型

In [ ]:
# 导入依赖
from pymilvus import MilvusClient, DataType
from sentence_transformers import SentenceTransformer

In [2]:
# 1. 原始数据
items= [
    {
        "type":"phone",
        "id":"用户A",
        "content":"用户A的手机号是13800001234"
    },
    {
        "type":"phone",
        "id":"用户B",
        "content":"用户B的手机号是13800005678"
    },
    {
        "type":"order",
        "id":"订单1001",
        "content":"订单1001编号203011010001"
    },
    {
        "type":"order",
        "id":"订单1002",
        "content":"订单1002编号203011010123"
    }
]

In [ ]:
# 2. 加载Embedding模型
model = SentenceTransformer(r"D:\models\bge-small-zh-v1.5")       # SentenceTransformer :句子向量化模型库

texts = [
    iterm["content"] for iterm in items             # items 里只抽出 content 字段，组成一个纯文本列表
]

embeddings = model.encode(                          # 把一批文本批量转成向量
    texts,
    normalize_embeddings=True                       # 把每个向量归一化（长度变成 1,拿到余弦）
)

dimension = len(embeddings[0])      # 取第一条向量的长度 = 向量维度(因为 Milvus 建表时必须明确告诉它向量是几维的)

print("向量维度：", dimension)

In [ ]:
# 3. 连接Milvus
client = MilvusClient(
    uri = "http://localhost:19530"
)

collection_name = "knowledge_demo"      # 命名空间

In [ ]:
# 4. 建 Schema (定义表结构,告诉 Milvus 这个表里有哪些字段,每个字段是什么类型)
schema = client.create_schema()

schema.add_field(
    field_name="id",               # 主键字段
    datatype=DataType.INT64,       # 类型
    is_primary=True,               # 主键
    auto_id=True                   # 自动递增
)
schema.add_field(
    field_name="type",             # 类型字段
    datatype=DataType.VARCHAR,
    max_length=50
)
schema.add_field(
    field_name="item_id",          # 项目ID字段
    datatype=DataType.VARCHAR,
    max_length=100
)
schema.add_field(
    field_name="content",          # 文本字段
    datatype=DataType.VARCHAR,
    max_length=500
)
schema.add_field(
    field_name="embedding",        # 存向量字段
    datatype=DataType.FLOAT_VECTOR,
    dim=dimension
)

# 执行建表,创建集合knowledge_demo(这样Milvus里就有了一个空表，结构齐全，但没数据)
if not client.has_collection(collection_name):
    client.create_collection(collection_name, schema=schema) 

In [ ]:
# 5. 插入数据
data = []
# zip:把「原始数据」和「向量」两表一一对应,配对合并成 Milvus 要的格式
for item,vector in zip(items, embeddings):
    data.append(
        {
            "type":item["type"],
            "item_id":item["id"],
            "content":item["content"],
            "embedding":vector.tolist()   #把 numpy 数组转成Python列表(Milvus客户端接收的是普通list，不认 numpy 数组)
        }
    )

In [ ]:
# 6. 批量插入集合
client.insert(collection_name, data)    # 之前把insert换成upsert,为防止重复数据自动更新，但是这里的id是自增的，正常用insert就行，用了upsert反而错
client.flush(collection_name)           # ← 强制落盘，把内存数据写到磁盘，防止刚插的数据搜不到

In [ ]:
# 7. 创建索引：已存在就跳过
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="embedding",    # 给哪个字段建索引
    index_type="AUTOINDEX",    # 用什么算法建索引，AUTOINDEX 是 Milvus 自动选算法
    metric_type="COSINE"       # 比相似度用什么公式，COSINE 是余弦相似度
)
# 如果索引不存在，才创建
if not client.list_indexes(collection_name):
    client.create_index(collection_name, index_params)

In [ ]:
client.load_collection(collection_name)   # ← 加载到内存才能搜
# 8. 查询
query_text="手机号13800001234是谁"

query_vector = model.encode(            # 2和7呼应：分别是存和查，必须保证用同一个模型、同一种归一化方式，存和查才能比
    [query_text],                       # 把查询文本转换为向量
    normalize_embeddings=True           # 归一化向量，确保向量长度为 1
)

result=client.search(
    collection_name,                    # 搜索哪个集合（3、4定义）
    data=query_vector.tolist(),         # 查询向量（转成 list，2、5定义）
    limit=3,                            # 返回 3 条结果 
    output_fields=[                     # 定义返回字段（4）
        "type",
        "item_id",
        "content"
    ],
    search_params={
        "metric_type":"COSINE"          # 搜索时的度量方式(必须和第 6 步建索引时的 metric_type 一致，否则 Milvus 报错)
    }
)

for hits in result:
    for hit in hits:
        print("实体:",hit["entity"])
        print("距离:",hit["distance"])


距离: 0.8499037623405457
实体: {'type': 'phone', 'item_id': '用户A', 'content': '用户A的手机号是13800001234'}
距离: 0.7996397614479065
实体: {'type': 'phone', 'item_id': '用户B', 'content': '用户B的手机号是13800005678'}
距离: 0.563514232635498
实体: {'type': 'order', 'item_id': '订单1001', 'content': '订单1001编号203011010001'}
